# Split Datasets

Split dataset to Train/Validation/Test sets

## A. Overview

Random split to Train, Validation and Test sets

## B. Combine Datasets

In [1]:
from pathlib import Path
import csv
import os
import random
import json

from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import configuration
from src import data_utils, setup

RANDS_STATE = 42
random.seed(RANDS_STATE)

dataset_path = Path('..') / 'data'

### B.1. Load Datasets

In [2]:
# disaster_frac = data_utils.get_data_disaster_fraction()
# disaster_filepath = dataset_path / 'disaster' / str(disaster_frac)

# df_disaster_informative = pd.read_csv(disaster_filepath / 'informative.csv')
# df_disaster_informative['subset'] = 'disaster'
# # df_disaster_humanitarian = pd.read_csv(disaster_filepath / 'humanitarian.csv')
# # df_disaster_humanitarian['subset'] = 'disaster'

In [3]:
extended_filepath = dataset_path / "extended"

df_weather = pd.read_csv(
    extended_filepath / "weather.csv"
)["tweet_text"].to_frame()
df_weather["informative"] = False
df_weather["subset"] = "weather"

# df_out_topic = pd.read_csv(
#     extended_filepath / "out_topic.csv"
# )["tweet_text"].to_frame()
# df_out_topic["informative"] = False
# df_out_topic["subset"] = "out_topic"

### B.2. Combine Datasets

In [4]:
# df_informative = (
#     pd.concat([df_disaster_informative, df_weather, df_out_topic], ignore_index=True)
#     .sample(frac=1, random_state=setup.RANDOM_SEED)
#     .reset_index(drop=True)
# )
# df_informative.head()

## C. Filtering

### C.1. Single tokens

In [5]:
df_weather.shape
# df_out_topic.shape

(47518, 3)

In [6]:
def filter_duplicates_with_resume(
    df,
    text_column="tweet_text",
    chunk_size=5000,
    similarity_threshold=0.75,
    checkpoint_file="checkpoint.json",
):
    print(f"Original dataset size: {len(df)}")

    # Bag-of-ngram vectorization
    print("Vectorizing text...")
    vectorizer = CountVectorizer(ngram_range=(1, 2))
    tfidf_matrix = vectorizer.fit_transform(df[text_column])

    num_rows = tfidf_matrix.shape[0]

    indices_to_drop = set()
    # For restore
    start_processing_row = 0

    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, "r") as f:
            state = json.load(f)
            start_processing_row = state.get('last_processed_row', 0)
            indices_to_drop = set(state.get('indices_to_drop', []))
        print(
            f"Resuming from row {start_processing_row}. Previously identified {len(indices_to_drop)} duplicates."
        )
    else:
        print("Starting fresh chunked cosine similarity...")

    # Apply Chunking to avoid Out-of-Memory issues with large datasets
    for start_row in range(start_processing_row, num_rows, chunk_size):
        end_row = min(start_row + chunk_size, num_rows)

        chunk_matrix = tfidf_matrix[start_row:end_row]
        target_matrix = tfidf_matrix[start_row:]

        similarities = cosine_similarity(chunk_matrix, target_matrix)
        x_indices, y_indices = np.where(similarities > similarity_threshold)

        for x, y in zip(x_indices, y_indices):
            actual_x = start_row + x
            actual_y = start_row + y

            if actual_x < actual_y and actual_x not in indices_to_drop:
                indices_to_drop.add(int(actual_y))

        # Save State to Checkpoint file
        state = {
            "last_processed_row": int(end_row),
            "indices_to_drop": list(indices_to_drop),  # JSON doesn't support sets
        }
        # to prevent corruption if interrupted exactly during the write operation
        # => Write to a temporary file first, then rename
        temp_checkpoint = f"{checkpoint_file}.tmp"
        with open(temp_checkpoint, "w") as f:
            json.dump(state, f)
        os.replace(temp_checkpoint, checkpoint_file)

        print(
            f"Processed up to row {end_row}/{num_rows}. Identified {len(indices_to_drop)} near-duplicates in total."
        )

    df_filtered = df.drop(index=list(indices_to_drop)).reset_index(drop=True)

    # Remove checkpoint file upon successful completion
    if os.path.exists(checkpoint_file):
        os.remove(checkpoint_file)
        print("Processing complete. Checkpoint file removed.")

    print(f"Final dataset size after near-duplicate removal: {len(df_filtered)}")
    return df_filtered

In [7]:
df_weather_processed = filter_duplicates_with_resume(df_weather) # 41414

Original dataset size: 47518
Vectorizing text...
Starting fresh chunked cosine similarity...
Processed up to row 5000/47518. Identified 1412 near-duplicates in total.
Processed up to row 10000/47518. Identified 2197 near-duplicates in total.
Processed up to row 15000/47518. Identified 2768 near-duplicates in total.
Processed up to row 20000/47518. Identified 3734 near-duplicates in total.
Processed up to row 25000/47518. Identified 4317 near-duplicates in total.
Processed up to row 30000/47518. Identified 4777 near-duplicates in total.
Processed up to row 35000/47518. Identified 5419 near-duplicates in total.
Processed up to row 40000/47518. Identified 5971 near-duplicates in total.
Processed up to row 45000/47518. Identified 6078 near-duplicates in total.
Processed up to row 47518/47518. Identified 6104 near-duplicates in total.
Processing complete. Checkpoint file removed.
Final dataset size after near-duplicate removal: 41414
